# B2.5 · Model tiering and routing inside the loop

**Function B — Product & Application Security → The Security Automation / Harness Engineer**  ·  *AI for Security*

Builds on **[B2.4 · Budgets and stop conditions](https://spbreed.github.io/cyber-commons/lessons/B2.4.html)**.

| | |
|---|---|
| Open-source tooling | LiteLLM, vLLM |
| Open-weight models | Llama 3.3, GLM-4.6, Kimi K2 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


A1.7 routed models across *stages* at the architecture level. This lesson routes
them **inside the loop**, where the decision is made per iteration and the
temptation is stronger.

The cost pressure is real: a large open-weight model on every iteration of a
50-step loop is slow and expensive, and most iterations are trivial. So teams
route dynamically — small model by default, escalate to the large one when the
task looks hard.

Two rules keep that safe, and they are the same two from A1.7 applied per-call:

1. **A model may only invoke tools within its tier's blast-radius budget.**
2. **The verifier is never weaker than the actor.**

The failure mode specific to in-loop routing is subtler than either: **escalation
on failure**. If the loop retries with a bigger model whenever the small one
fails verification, an attacker who can cause failures can force every task onto
the most capable model — and, more importantly, the escalation path usually
carries more authority too.

> **About the model in this notebook.** It runs offline against a deterministic
> replay, so the lesson executes on a Kaggle kernel with no network. The replay
> is not a language model and is labelled as such. To run the identical harness
> against a real open-weight model:
>
> ```bash
> ollama pull glm-4.6            # or kimi-k2, llama3.3
> export OPENAI_BASE_URL=http://localhost:11434/v1 OPENAI_API_KEY=ollama MODEL=glm-4.6
> ```

## 2 · Demo — tiered routing that works

In [ ]:
TIERS = {
 "llama3.2:1b":  {"tier": 0, "ms": 40,   "solves": 0.2},
 "llama3.3:8b":  {"tier": 1, "ms": 220,  "solves": 0.55},
 "glm-4.6":      {"tier": 2, "ms": 900,  "solves": 0.85},
 "kimi-k2":      {"tier": 3, "ms": 2400, "solves": 0.93},
}
TIER_BUDGET = {0: 0, 1: 3, 2: 20, 3: 60}     # max blast radius a tier may hold
SCOPE = {"read_file": 0, "search": 0, "write_file": 3, "open_pr": 3,
         "merge_pr": 6, "deploy": 40}

def may_invoke(model, tool):
    return SCOPE[tool] <= TIER_BUDGET[TIERS[model]["tier"]]

print(f"{'model':14s}{'tier':>5}  tools it may invoke")
print("-" * 70)
for m in TIERS:
    allowed = [t for t in SCOPE if may_invoke(m, t)]
    print(f"{m:14s}{TIERS[m]['tier']:>5}  {allowed}")

## 3 · Where it breaks — escalation on failure

The natural retry policy: if the small model fails, try a bigger one. Watch what an attacker who can force failures gets.

In [ ]:
LADDER = ["llama3.2:1b", "llama3.3:8b", "glm-4.6", "kimi-k2"]

def loop_with_escalation(task_fails_always, max_steps=4, escalate_authority=True):
    """The common pattern: harder task → bigger model → and, usually, more tools."""
    trace = []
    for i in range(max_steps):
        model = LADDER[min(i, len(LADDER)-1)]
        tools = [t for t in SCOPE if may_invoke(model, t)] if escalate_authority \
                else [t for t in SCOPE if may_invoke(LADDER[0], t)]
        trace.append({"step": i+1, "model": model, "tier": TIERS[model]["tier"],
                      "ms": TIERS[model]["ms"], "tools": tools})
        if not task_fails_always:
            break
    return trace

print("a task that keeps failing verification:")
tr = loop_with_escalation(task_fails_always=True)
for s in tr:
    print(f"   step {s['step']}  {s['model']:14s} tier {s['tier']}  "
          f"{s['ms']:>5}ms  may invoke {s['tools']}")
total_ms = sum(s["ms"] for s in tr)
print(f"\ncost of one forced escalation: {total_ms}ms and the final step could "
      f"invoke {tr[-1]['tools']}")
print("An attacker who can make verification fail has just promoted the loop to")
print("the most capable model AND the widest tool set. Both, for free.")

## 4 · The control — escalate capability, never authority

The fix separates two things that are usually coupled: how *smart* the model is, and what it is *allowed to do*. Escalating the first is fine. Escalating the second must require a fresh decision.

In [ ]:
def loop_capability_only(task_fails_always, max_steps=4, task_budget=TIER_BUDGET[1]):
    """Authority is fixed by the TASK, not by which model happens to be running.

    The ladder starts at the lowest tier whose budget covers the task's
    authority. Routing a tool-holding step to a model below that would hand the
    weakest model in the system tools its tier is not trusted with — which is
    the same mistake as escalating authority, in the other direction.
    """
    ladder = [m for m in LADDER if TIER_BUDGET[TIERS[m]["tier"]] >= task_budget]
    trace = []
    for i in range(max_steps):
        model = ladder[min(i, len(ladder)-1)]
        tools = [t for t in SCOPE if SCOPE[t] <= task_budget]
        trace.append({"step": i+1, "model": model, "tools": tools})
        if not task_fails_always:
            break
    return trace

tr2 = loop_capability_only(task_fails_always=True)
print(f"   task authority budget: {TIER_BUDGET[1]}  → ladder starts at the lowest "
      f"tier that covers it")
for s in tr2:
    print(f"   step {s['step']}  {s['model']:14s} may invoke {s['tools']}")
print("\nThe model gets smarter. The authority does not move.")

escalated = set(tr[-1]["tools"]) - set(tr2[-1]["tools"])
print(f"tools the attacker gained under the naive policy: {sorted(escalated)}")
assert escalated

In [ ]:
# Verify: both rules, checked over every step of both policies.
def review(trace, verifier_model):
    problems = []
    for s in trace:
        model = s["model"]
        for t in s["tools"]:
            if SCOPE[t] > TIER_BUDGET[TIERS[model]["tier"]]:
                problems.append(f"step {s['step']}: {model} may invoke {t} "
                                f"(blast {SCOPE[t]} > budget "
                                f"{TIER_BUDGET[TIERS[model]['tier']]})")
        if TIERS[verifier_model]["tier"] < TIERS[model]["tier"]:
            problems.append(f"step {s['step']}: verifier {verifier_model} is weaker "
                            f"than actor {model}")
    return problems

for label, trace, verifier in (("escalate authority too",   tr,  "llama3.3:8b"),
                               ("capability only, glm verifier", tr2, "glm-4.6"),
                               ("capability only, kimi verifier", tr2, "kimi-k2")):
    p = review(trace, verifier)
    print(f"{label:32s} {'PASS' if not p else f'{len(p)} FINDING(S)'}")
    for x in p[:4]:
        print(f"      ⚠ {x}")

print("\nRead the middle row. Fixing the authority leak was not enough:")
print("once the ladder can reach kimi-k2, a glm-4.6 verifier is weaker than the")
print("actor on the final step, and rule 2 fires. Escalating capability forces")
print("the verifier's tier up with it — a second-order cost of dynamic routing")
print("that cost models never include.")

assert review(tr2, "glm-4.6"), "the weaker-verifier finding must be reported"
assert review(tr2, "kimi-k2") == [], "top-tier verifier should satisfy both rules"
top = max(TIERS[s["model"]]["tier"] for s in tr2)
print(f"\nrule: verifier tier must be ≥ {top} (the highest tier the ladder reaches)")

## What you just proved

The tier table shows only `kimi-k2` may invoke `deploy`. Under escalation-on-failure a forced failure walks the loop up to `kimi-k2` in 3,560ms and hands it `merge_pr` and `deploy`. The capability-only policy keeps the tool set fixed throughout. The review then shows a second-order effect: with a `glm-4.6` verifier the capability-only policy still fails rule 2 on the final step, because the ladder reached a stronger model than the verifier. Only a top-tier verifier passes both rules.

## Your turn

Check your own retry logic: when a step fails and you retry with a different model, does the tool set change? In most frameworks the answer is yes and nobody chose it.

---

**Next → [B2.6 · Sub-agents and delegation depth](https://spbreed.github.io/cyber-commons/lessons/B2.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*